# 🕸️ Web Scraping & Data Analysis Pipeline
### Complete Project Notebook — Yash

**Data Sources:**
- 📘 Site 1 — `quotes.toscrape.com` (static scraping — quotes, authors, tags)
- 📙 Site 2 — `books.toscrape.com` (multi-page scraping — books, prices, ratings)
- 🌐 API — `Open Library API` (book metadata — subjects, publish year, pages)

**Pipeline:** Scrape → Clean → Store (CSV + SQLite) → Analyze → Visualize


## ⚙️ Step 0 — Install & Import Libraries

In [ ]:
# Run once to install dependencies
# !pip install requests beautifulsoup4 pandas numpy matplotlib seaborn plotly sqlalchemy lxml

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sqlite3
import re
import time
import warnings
warnings.filterwarnings('ignore')

# ── Visual Style ──
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

print("✅ All libraries imported successfully!")


## 🌐 Step 1 — Scrape Site 1: quotes.toscrape.com
**Type:** Static HTML | **Tool:** requests + BeautifulSoup  
**Target Data:** Quote text, Author name, Tags, Author born date & location  
**Pages:** All 10 pages → ~100 quotes with author bio details


In [ ]:
BASE_URL = "https://quotes.toscrape.com"
HEADERS  = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

def scrape_author_bio(author_url):
    """Scrapes author born-date and born-location from their detail page."""
    try:
        r = requests.get(BASE_URL + author_url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        born_date = soup.select_one(".author-born-date")
        born_loc  = soup.select_one(".author-born-location")
        return (
            born_date.text.strip() if born_date else "Unknown",
            born_loc.text.strip().lstrip("in ") if born_loc else "Unknown"
        )
    except:
        return "Unknown", "Unknown"

quotes_records = []
visited_authors = {}   # cache so we don't re-scrape same author page
page = 1

print("🔄 Scraping quotes.toscrape.com ...")
while True:
    url = f"{BASE_URL}/page/{page}/"
    r   = requests.get(url, headers=HEADERS, timeout=10)
    if r.status_code != 200:
        break
    soup = BeautifulSoup(r.text, "html.parser")
    quote_divs = soup.select("div.quote")
    if not quote_divs:
        break

    for div in quote_divs:
        text   = div.select_one("span.text").text.strip().strip('“”')
        author = div.select_one("small.author").text.strip()
        tags   = [t.text for t in div.select("a.tag")]
        a_href = div.select_one("a[href*='/author/']")['href']

        if author not in visited_authors:
            visited_authors[author] = scrape_author_bio(a_href)
            time.sleep(0.2)   # be polite

        born_date, born_loc = visited_authors[author]
        quotes_records.append({
            "quote"     : text,
            "author"    : author,
            "tags"      : ", ".join(tags),
            "tag_count" : len(tags),
            "born_date" : born_date,
            "born_loc"  : born_loc,
            "source"    : "quotes.toscrape.com"
        })

    print(f"  Page {page} ✓ — {len(quote_divs)} quotes")
    next_btn = soup.select_one("li.next a")
    if not next_btn:
        break
    page += 1

df_quotes = pd.DataFrame(quotes_records)
print(f"\n✅ Total Quotes Scraped: {len(df_quotes)}")
df_quotes.head(3)


## 📚 Step 2 — Scrape Site 2: books.toscrape.com
**Type:** Multi-page Static HTML | **Tool:** requests + BeautifulSoup  
**Target Data:** Title, Price, Rating, Availability, Category  
**Pages:** All 50 pages → 1000 books ✅ (exceeds 500 record requirement)


In [ ]:
BOOKS_BASE = "https://books.toscrape.com"
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

books_records = []
print("🔄 Scraping books.toscrape.com ...")

for page in range(1, 51):   # 50 pages × 20 books = 1000 records
    url  = f"{BOOKS_BASE}/catalogue/page-{page}.html"
    r    = requests.get(url, headers=HEADERS, timeout=10)
    if r.status_code != 200:
        print(f"  Stopped at page {page} (status {r.status_code})")
        break
    soup = BeautifulSoup(r.text, "html.parser")

    for article in soup.select("article.product_pod"):
        title  = article.select_one("h3 a")["title"]
        price_raw = article.select_one(".price_color").text.strip()
        price  = float(re.sub(r"[^\d.]", "", price_raw))
        rating_word = article.select_one(".star-rating")["class"][1]
        rating = RATING_MAP.get(rating_word, 0)
        avail  = article.select_one(".availability").text.strip()

        books_records.append({
            "title"       : title,
            "price_gbp"   : price,
            "rating"      : rating,
            "availability": avail,
            "source"      : "books.toscrape.com"
        })

    if page % 10 == 0:
        print(f"  Page {page}/50 ✓ — {len(books_records)} books so far")

df_books = pd.DataFrame(books_records)
print(f"\n✅ Total Books Scraped: {len(df_books)}")
df_books.head(3)


## 🌍 Step 3 — Public API Integration: Open Library API
**API:** `https://openlibrary.org/search.json`  
**No API key required** ✅  
**Data:** Books with subjects, publish year, number of pages, language, edition count  
We query multiple subjects to get 500+ API records.


In [ ]:
OL_BASE   = "https://openlibrary.org/search.json"
SUBJECTS  = ["fiction", "science", "history", "philosophy", "technology",
             "psychology", "biography", "art", "economics", "politics"]

api_records = []
print("🔄 Fetching from Open Library API ...")

for subject in SUBJECTS:
    params = {"subject": subject, "limit": 80, "fields": "title,author_name,subject,first_publish_year,number_of_pages_median,language,edition_count"}
    try:
        r    = requests.get(OL_BASE, params=params, headers=HEADERS, timeout=15)
        data = r.json()
        docs = data.get("docs", [])
        for doc in docs:
            api_records.append({
                "title"           : doc.get("title", "Unknown"),
                "author"          : ", ".join(doc.get("author_name", ["Unknown"])[:2]),
                "first_pub_year"  : doc.get("first_publish_year", np.nan),
                "pages"           : doc.get("number_of_pages_median", np.nan),
                "edition_count"   : doc.get("edition_count", np.nan),
                "language"        : doc.get("language", ["unknown"])[0] if doc.get("language") else "unknown",
                "query_subject"   : subject,
                "source"          : "openlibrary_api"
            })
        print(f"  Subject '{subject}': {len(docs)} records fetched")
        time.sleep(0.3)
    except Exception as e:
        print(f"  Subject '{subject}': ERROR — {e}")

df_api = pd.DataFrame(api_records)
print(f"\n✅ Total API Records: {len(df_api)}")
df_api.head(3)


## 🧹 Step 4 — Data Cleaning & Preprocessing
Applying cleaning to each dataset individually, then merging into one master dataframe.


In [ ]:
print("=" * 55)
print("BEFORE CLEANING")
print("=" * 55)
print(f"Quotes  rows: {len(df_quotes)}  | nulls: {df_quotes.isnull().sum().sum()}")
print(f"Books   rows: {len(df_books)}   | nulls: {df_books.isnull().sum().sum()}")
print(f"API     rows: {len(df_api)}     | nulls: {df_api.isnull().sum().sum()}")


In [ ]:
# ── Clean df_quotes ──────────────────────────────────────────────
df_q = df_quotes.copy()
df_q.drop_duplicates(subset=["quote"], inplace=True)
df_q["quote"]      = df_q["quote"].str.strip()
df_q["author"]     = df_q["author"].str.strip().str.title()
df_q["tags"]       = df_q["tags"].fillna("untagged")
df_q["born_date"]  = df_q["born_date"].replace("Unknown", np.nan)
df_q["born_loc"]   = df_q["born_loc"].replace("Unknown", np.nan)
df_q["quote_len"]  = df_q["quote"].str.len()    # derived feature

print(f"✅ Quotes after cleaning: {len(df_q)} rows")


In [ ]:
# ── Clean df_books ───────────────────────────────────────────────
df_b = df_books.copy()
df_b.drop_duplicates(subset=["title"], inplace=True)
df_b["title"]        = df_b["title"].str.strip()
df_b["availability"] = df_b["availability"].str.strip()
df_b["in_stock"]     = df_b["availability"].apply(lambda x: 1 if "In stock" in x else 0)
df_b["price_inr"]    = (df_b["price_gbp"] * 107).round(2)   # GBP → INR approx
df_b["price_tier"]   = pd.cut(df_b["price_gbp"],
                               bins=[0, 20, 40, 60, 100],
                               labels=["Budget", "Mid", "Premium", "Luxury"])

print(f"✅ Books after cleaning: {len(df_b)} rows")
df_b[["title","price_gbp","price_inr","price_tier","rating","in_stock"]].head(3)


In [ ]:
# ── Clean df_api ─────────────────────────────────────────────────
df_a = df_api.copy()
df_a.drop_duplicates(subset=["title", "author"], inplace=True)
df_a["title"]         = df_a["title"].str.strip()
df_a["author"]        = df_a["author"].fillna("Unknown").str.title()

# Filter obviously bad years
current_year = 2025
df_a = df_a[(df_a["first_pub_year"].isna()) |
            ((df_a["first_pub_year"] >= 1800) & (df_a["first_pub_year"] <= current_year))]

df_a["pages"]         = pd.to_numeric(df_a["pages"], errors="coerce")
df_a["pages"]         = df_a["pages"].clip(upper=5000)      # remove absurd outliers
df_a["edition_count"] = pd.to_numeric(df_a["edition_count"], errors="coerce")
df_a["decade"]        = (df_a["first_pub_year"] // 10 * 10).astype("Int64")
df_a["language"]      = df_a["language"].str.lower().str.strip()

print(f"✅ API data after cleaning: {len(df_a)} rows")
df_a.head(3)


In [ ]:
print("=" * 55)
print("AFTER CLEANING SUMMARY")
print("=" * 55)
print(f"Quotes  rows: {len(df_q)}  | nulls: {df_q[['quote','author','tags']].isnull().sum().sum()}")
print(f"Books   rows: {len(df_b)} | nulls: {df_b[['title','price_gbp','rating']].isnull().sum().sum()}")
print(f"API     rows: {len(df_a)} | nulls: {df_a[['title','author']].isnull().sum().sum()}")
print(f"\n📦 TOTAL RECORDS: {len(df_q) + len(df_b) + len(df_a)}")


## 💾 Step 5 — Store Data in CSV + SQLite Database

In [ ]:
import os
os.makedirs("output", exist_ok=True)

# ── Save CSVs ────────────────────────────────────────────────────
df_q.to_csv("output/quotes_cleaned.csv",  index=False)
df_b.to_csv("output/books_cleaned.csv",   index=False)
df_a.to_csv("output/api_books_cleaned.csv", index=False)
print("✅ CSVs saved in /output/")

# ── Save to SQLite ───────────────────────────────────────────────
conn = sqlite3.connect("output/project_data.db")

df_q.to_sql("quotes",    conn, if_exists="replace", index=False)
df_b.to_sql("books",     conn, if_exists="replace", index=False)
df_a.to_sql("api_books", conn, if_exists="replace", index=False)

conn.commit()
print("✅ SQLite DB saved → output/project_data.db")

# ── Verify DB ────────────────────────────────────────────────────
cursor = conn.cursor()
for table in ["quotes", "books", "api_books"]:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"  Table '{table}': {count} rows")
conn.close()


## 📊 Step 6 — Data Analysis & Insights

In [ ]:
print("━" * 55)
print("INSIGHT 1 — Books: Price & Rating Statistics")
print("━" * 55)
print(df_b[["price_gbp", "price_inr", "rating"]].describe().round(2))


In [ ]:
print("━" * 55)
print("INSIGHT 2 — Rating Distribution")
print("━" * 55)
rating_dist = df_b["rating"].value_counts().sort_index()
for r, c in rating_dist.items():
    stars = "★" * r + "☆" * (5 - r)
    print(f"  {stars}  : {c} books ({c/len(df_b)*100:.1f}%)")


In [ ]:
print("━" * 55)
print("INSIGHT 3 — Average Price by Rating")
print("━" * 55)
avg_price_by_rating = df_b.groupby("rating")["price_gbp"].mean().round(2)
print(avg_price_by_rating.to_string())
print("\n→ Higher rated books tend to be", 
      "cheaper" if avg_price_by_rating.corr(pd.Series(avg_price_by_rating.index, index=avg_price_by_rating.index)) < 0 else "pricier")


In [ ]:
print("━" * 55)
print("INSIGHT 4 — Top 10 Most Quoted Authors")
print("━" * 55)
top_authors = df_q["author"].value_counts().head(10)
print(top_authors.to_string())


In [ ]:
print("━" * 55)
print("INSIGHT 5 — Top 15 Most Used Tags")
print("━" * 55)
all_tags = df_q["tags"].str.split(", ").explode()
top_tags = all_tags.value_counts().head(15)
print(top_tags.to_string())


In [ ]:
print("━" * 55)
print("INSIGHT 6 — API Books: Publication Decades")
print("━" * 55)
decade_counts = df_a["decade"].dropna().astype(int).value_counts().sort_index()
print(decade_counts.to_string())


In [ ]:
print("━" * 55)
print("INSIGHT 7 — Top Subjects by Edition Count")
print("━" * 55)
top_subjects = df_a.groupby("query_subject")["edition_count"].mean().round(1).sort_values(ascending=False)
print(top_subjects.to_string())
print("\n→ Subject with most editions:", top_subjects.idxmax())


In [ ]:
print("━" * 55)
print("INSIGHT 8 — Price Tier Distribution")
print("━" * 55)
tier_dist = df_b["price_tier"].value_counts()
for tier, count in tier_dist.items():
    print(f"  {tier:10s}: {count} books ({count/len(df_b)*100:.1f}%)")


## 📈 Step 7 — Visualizations (5 Plots + 2 Bonus)
> Each chart is saved as a PNG in `/output/`


### 📊 Visualization 1 — Rating Distribution (Books)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Books — Rating Analysis", fontsize=15, fontweight="bold", y=1.01)

# Left: count bar
colors = ["#e74c3c","#e67e22","#f1c40f","#2ecc71","#1abc9c"]
rating_dist.plot(kind="bar", ax=axes[0], color=colors, edgecolor="white", linewidth=1.2)
axes[0].set_title("Number of Books per Star Rating", fontweight="bold")
axes[0].set_xlabel("Star Rating")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels(["★","★★","★★★","★★★★","★★★★★"], rotation=0, fontsize=13)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x()+p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=11)

# Right: avg price per rating
avg_price_by_rating.plot(kind="bar", ax=axes[1], color="#3498db", edgecolor="white")
axes[1].set_title("Average Price (£) per Star Rating", fontweight="bold")
axes[1].set_xlabel("Star Rating")
axes[1].set_ylabel("Avg Price (£)")
axes[1].set_xticklabels(["★","★★","★★★","★★★★","★★★★★"], rotation=0, fontsize=13)
for p in axes[1].patches:
    axes[1].annotate(f'£{p.get_height():.1f}', (p.get_x()+p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig("output/viz1_rating_distribution.png", bbox_inches="tight", dpi=150)
plt.show()
print("✅ Saved: output/viz1_rating_distribution.png")


### 📊 Visualization 2 — Price Tier Distribution (Pie + Box)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Books — Price Analysis", fontsize=15, fontweight="bold")

# Left: Pie
pie_colors = ["#3498db","#2ecc71","#e74c3c","#9b59b6"]
wedges, texts, autotexts = axes[0].pie(
    tier_dist.values, labels=tier_dist.index,
    autopct='%1.1f%%', colors=pie_colors,
    startangle=140, pctdistance=0.82,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for t in autotexts:
    t.set_fontsize(11)
axes[0].set_title("Price Tier Distribution", fontweight="bold")

# Right: Box plot per rating
rating_groups = [df_b[df_b["rating"] == r]["price_gbp"].values for r in range(1, 6)]
bp = axes[1].boxplot(rating_groups, labels=["★","★★","★★★","★★★★","★★★★★"],
                     patch_artist=True, notch=False,
                     medianprops=dict(color="red", linewidth=2))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title("Price Distribution by Rating (Box Plot)", fontweight="bold")
axes[1].set_xlabel("Star Rating")
axes[1].set_ylabel("Price (£)")

plt.tight_layout()
plt.savefig("output/viz2_price_analysis.png", bbox_inches="tight", dpi=150)
plt.show()
print("✅ Saved: output/viz2_price_analysis.png")


### 📊 Visualization 3 — Top 10 Most Quoted Authors

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

palette = sns.color_palette("coolwarm", len(top_authors))
bars = ax.barh(top_authors.index[::-1], top_authors.values[::-1],
               color=palette, edgecolor="white", linewidth=0.8)

for bar, val in zip(bars, top_authors.values[::-1]):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f' {val} quotes', va='center', fontsize=11)

ax.set_title("Top 10 Most Quoted Authors", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Number of Quotes", fontsize=12)
ax.set_xlim(0, top_authors.max() + 2.5)
ax.spines[["top","right","left"]].set_visible(False)
ax.tick_params(left=False)
ax.set_yticklabels(top_authors.index[::-1], fontsize=11)

plt.tight_layout()
plt.savefig("output/viz3_top_authors.png", bbox_inches="tight", dpi=150)
plt.show()
print("✅ Saved: output/viz3_top_authors.png")


### 📊 Visualization 4 — Top 15 Quote Tags (Horizontal Bar)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

cmap   = plt.cm.get_cmap("viridis", len(top_tags))
colors_tags = [cmap(i) for i in range(len(top_tags))]
bars = ax.barh(top_tags.index[::-1], top_tags.values[::-1],
               color=colors_tags[::-1], edgecolor="white")

for bar, val in zip(bars, top_tags.values[::-1]):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2,
            f' {val}', va='center', fontsize=10)

ax.set_title("Top 15 Most Used Tags on Quotes", fontsize=14, fontweight="bold")
ax.set_xlabel("Frequency", fontsize=12)
ax.spines[["top","right","left"]].set_visible(False)
ax.tick_params(left=False)

plt.tight_layout()
plt.savefig("output/viz4_top_tags.png", bbox_inches="tight", dpi=150)
plt.show()
print("✅ Saved: output/viz4_top_tags.png")


### 📊 Visualization 5 — Publication Decade Trend (API Data)

In [ ]:
decade_plot = decade_counts[decade_counts.index >= 1900]   # focus on modern era

fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(decade_plot.index, decade_plot.values, alpha=0.25, color="#e74c3c")
ax.plot(decade_plot.index, decade_plot.values, marker="o", color="#e74c3c",
        linewidth=2.5, markersize=7)

for x, y in zip(decade_plot.index, decade_plot.values):
    ax.annotate(f'{y}', (x, y), textcoords="offset points",
                xytext=(0, 8), ha='center', fontsize=9)

ax.set_title("Books Published per Decade (Open Library API)", fontsize=14, fontweight="bold")
ax.set_xlabel("Decade", fontsize=12)
ax.set_ylabel("Number of Books in Dataset", fontsize=12)
ax.set_xticks(decade_plot.index)
ax.set_xticklabels([f"{int(d)}s" for d in decade_plot.index], rotation=45, ha="right")
ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("output/viz5_publication_decade.png", bbox_inches="tight", dpi=150)
plt.show()
print("✅ Saved: output/viz5_publication_decade.png")


### 📊 Visualization 6 (Bonus) — Heatmap: Avg Edition Count by Subject

In [ ]:
pivot_lang = df_a[df_a["language"].isin(["eng","fre","spa","ger","por"])] \
                .groupby(["query_subject","language"])["edition_count"] \
                .mean().round(1).unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(pivot_lang, annot=True, fmt=".1f", cmap="YlOrRd",
            linewidths=0.5, linecolor="white", ax=ax,
            cbar_kws={"label": "Avg Edition Count"})
ax.set_title("Average Edition Count — Subject × Language (API Data)",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Language Code", fontsize=11)
ax.set_ylabel("Subject", fontsize=11)
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig("output/viz6_heatmap_subject_language.png", bbox_inches="tight", dpi=150)
plt.show()
print("✅ Saved: output/viz6_heatmap_subject_language.png")


### 📊 Visualization 7 (Bonus) — Interactive Scatter (Plotly)

In [ ]:
fig_px = px.scatter(
    df_b.sample(min(500, len(df_b)), random_state=42),
    x="price_gbp", y="rating",
    color="price_tier",
    size="price_gbp",
    hover_name="title",
    title="📚 Books — Price vs Rating (Interactive)",
    labels={"price_gbp": "Price (£)", "rating": "Star Rating",
            "price_tier": "Price Tier"},
    template="plotly_white",
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig_px.update_traces(marker=dict(opacity=0.75, line=dict(width=0.5, color="white")))
fig_px.update_layout(title_font_size=16, legend_title_text="Price Tier")
fig_px.write_html("output/viz7_interactive_scatter.html")
fig_px.show()
print("✅ Saved: output/viz7_interactive_scatter.html  (open in browser for interactivity)")


## ✅ Step 8 — Final Summary

In [ ]:
print("=" * 60)
print("         PROJECT COMPLETION SUMMARY")
print("=" * 60)
total = len(df_q) + len(df_b) + len(df_a)
print(f"  ✅ Site 1 Scraped  : quotes.toscrape.com   — {len(df_q)} records")
print(f"  ✅ Site 2 Scraped  : books.toscrape.com    — {len(df_b)} records")
print(f"  ✅ API Integrated  : Open Library API      — {len(df_a)} records")
print(f"  ✅ Total Records   : {total} (500+ ✓)")
print()
print(f"  ✅ Data Cleaning   : Duplicates, nulls, type-casting, derived features")
print(f"  ✅ CSV Files       : quotes_cleaned.csv, books_cleaned.csv, api_books_cleaned.csv")
print(f"  ✅ Database        : output/project_data.db (SQLite, 3 tables)")
print(f"  ✅ Insights        : 8 meaningful analytical insights generated")
print(f"  ✅ Visualizations  : 7 charts (5 required + 2 bonus) saved to /output/")
print("=" * 60)
print("\n📁 All output files are in the /output/ folder")
print("   Open viz7_interactive_scatter.html in browser for interactive chart!")
